# **Step (2) : Data Cleaning**

In [33]:
import os
import re
import unicodedata
import warnings
from pathlib import Path
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display , HTML
import re
import unicodedata
import html
from spellchecker import SpellChecker




In [34]:
data_frame = pd.read_csv("../data-set/World-food-waste.csv")

In [35]:
def normalize_name(value):
    """Normalize a single name."""
    if pd.isna(value):
        return ""

    return " ".join(str(value).strip().casefold().split())


def normalize_code(value):
    """Normalize a single code."""
    if pd.isna(value):
        return ""

    value = str(value).strip()

    if value.endswith(".0"):
        value = value[:-2]

    if value.isdigit():
        return value.zfill(3)

    return value


def normalize_un_name(series):
    """Normalize UN names using Pandas vectorization."""
    return (
        series.astype("string")
        .str.strip()
        .str.casefold()
        .str.replace(r"\s+", " ", regex=True)
    )


def normalize_un_code(series):
    """Normalize UN codes using Pandas vectorization."""
    return (
        series.astype("string")
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
        .str.zfill(3)
    )


def download(url):
    response = requests.get(
        url,
        timeout=30,
        headers={"User-Agent": "Mozilla/5.0"}
    )
    response.raise_for_status()
    return response



In [36]:
df = data_frame.copy()

In [37]:
# [ADDED] 1) remove exact duplicate rows (180 in the raw file)
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print("Duplicate rows removed:", before - len(df))

# [ADDED] 2) keep the fact that loss was reported as a range (e.g. "1-2%") before loss_percentage_original is dropped
df["loss_is_range"] = (
    df["loss_percentage_original"].astype("string")
    .str.contains(r"\d\s*-\s*\d", regex=True)
    .fillna(False)
)
print("Range-reported rows:", int(df["loss_is_range"].sum()))

Duplicate rows removed: 180
Range-reported rows: 757


In [38]:
df.head()

,m49_code,country,region,cpc_code,commodity,year,loss_percentage,loss_percentage_original,loss_quantity,activity,food_supply_stage,treatment,cause_of_loss,sample_size,method_data_collection,reference,url,notes,loss_is_range
0,1,World,NaN,0111,Wheat,2022,1.0,1%,NaN,Processing,Processing,NaN,NaN,NaN,Survey,"Aintzane Esturo, (2022), Cereals Extraction Ra...",NaN,Derived product is wheat flour,False
1,1,World,NaN,0112,Maize (corn),2022,0.5,0.5%,NaN,Processing,Processing,NaN,NaN,NaN,Survey,"Aintzane Esturo, (2022), Cereals Extraction Ra...",NaN,Derived product is maize flour.,False
2,1,World,NaN,0112,Maize (corn),2022,2.0,2%,NaN,Processing,Processing,NaN,NaN,NaN,Survey,"Aintzane Esturo, (2022), Cereals Extraction Ra...",NaN,Derived product is Maize (corn) grits.,False
3,1,World,NaN,0112,Maize (corn),2022,2.0,2%,NaN,Processing,Processing,NaN,NaN,NaN,Survey,"Aintzane Esturo, (2022), Cereals Extraction Ra...",NaN,Derived product is Maize (corn) grits flaked.,False
4,1,World,NaN,0112,Maize (corn),2022,0.1,0.1%,NaN,Processing,Processing,NaN,NaN,NaN,Survey,"Aintzane Esturo, (2022), Cereals Extraction Ra...",NaN,Derived product is corn starch.,False


## **1. country column**

### **1. Country Name Standardization & M49 Code Validation**

In [39]:


country_mapping = {
    # Taiwan
    "Taiwan, Province of China": "Taiwan",
    "China,Taiwan": "Taiwan",
    # Netherlands
    "Netherlands (Kingdom of the)": "Netherlands",
    # North Macedonia
    "The former Yugoslav Republic of Macedonia": "North Macedonia",
    # United Kingdom
    "United Kingdom of Great Britain and Northern Ireland": "United Kingdom",
}

df["country"] = (
    df["country"]
    .replace(country_mapping)
)

In [40]:
country_name_mapping = {
    158: "Taiwan",
    180: "DR Congo",
    364: "Iran",
    384: "Ivory Coast",
    408: "North Korea",
    410: "South Korea",
    418: "Laos",
    419: "Latin America and the Caribbean",
    53: "Australia and New Zealand",
    659: "Saint Kitts and Nevis",
    68: "Bolivia",
    780: "Trinidad and Tobago",
    807: "North Macedonia",
    826: "UK",
    834: "Tanzania",
    840: "USA",
    862: "Venezuela",
}

df["country"] = (
    df["m49_code"]
    .map(lambda code: normalize_code(code))
    .map(lambda code: country_name_mapping.get(int(code), None)
         if pd.notna(code) and code.isdigit()
         else None)
    .fillna(df["country"])
)

In [41]:

result = df.groupby("m49_code")["country"].unique().reset_index()

display(HTML(result.to_html(index=False)))

m49_code,country
1,[World]
2,[Africa]
4,[Afghanistan]
8,[Albania]
11,[Western Africa]
12,[Algeria]
15,[Northern Africa]
21,[Northern America]
24,[Angola]
31,[Azerbaijan]


### **2. Remove unknown country**

In [42]:
df = df[df["country"] != "Israel"].copy()

### **3. Handle area with regions**

In [43]:
# =========================================================
# 1. Load the saved classification CSV
# =========================================================

classification_file = (
    "../report/country/countries_regions_classification.csv"
)

classification = pd.read_csv(
    classification_file,
    dtype={"code_key": "string"}
)

# =========================================================
# 2. Prepare lookup from the CSV
# =========================================================

classification_lookup = (
    classification[
        ["code_key", "record_type"]
    ]
    .drop_duplicates(subset=["code_key"])
)

# =========================================================
# 3. Prepare original dataset
# =========================================================

df["name_key"] = normalize_un_name(df["country"])

df["code_key"] = normalize_un_code(df["m49_code"])

# =========================================================
# 4. Add record_type from the CSV
# [CHANGED] match on M49 code only: country names were renamed above (UK, USA, Iran...),
# so name+code matching silently dropped ~4,000 rows of real countries.
# =========================================================

df = df.merge(
    classification_lookup,
    on=["code_key"],
    how="left",
    validate="many_to_one"
)

# =========================================================
# 5. Review unclassified records
# =========================================================

unclassified = df[
    df["record_type"].isna()
    | df["record_type"].eq("Unclassified")
]

print("Unclassified records:", len(unclassified))

display(
    unclassified[["country", "m49_code"]]
    .drop_duplicates()
)

# =========================================================
# 6. Remove temporary columns
# =========================================================

df.drop(
    columns=["name_key", "code_key"],
    inplace=True
)

# =========================================================
# 7. Keep only Countries / Areas
# =========================================================

df = df[
    df["record_type"] == "Country / Area"
].copy()

# =========================================================
# 8. Display summary
# =========================================================

print("Clean country records:", len(df))

display(df["record_type"].value_counts())

display(df.head())




Unclassified records: 574


,country,m49_code
2214,Taiwan,158
6889,Palestine,275
10852,Ivory Coast,384
17905,Netherlands,528
25839,Turkey,792
26835,North Macedonia,807


Clean country records: 30590


record_type
Country / Area    30590
Name: count, dtype: int64

,m49_code,country,region,cpc_code,commodity,year,loss_percentage,loss_percentage_original,loss_quantity,activity,food_supply_stage,treatment,cause_of_loss,sample_size,method_data_collection,reference,url,notes,loss_is_range,record_type
29,104,Myanmar,NaN,23161.02,"Rice, milled",2015,5.88,5.88%,44.09kgs,Storage,Storage,"30 days storage, no trapping",Rodents,NaN,Controlled Experiment,Steven R. Belmain; Nyo Me Htwe; Nazira Q. Kama...,NaN,NaN,False,Country / Area
30,104,Myanmar,NaN,23161.02,"Rice, milled",2015,11.77,11.77%,88.18kgs,Storage,Storage,"60 days storage, no trapping",Rodents,NaN,Controlled Experiment,Steven R. Belmain; Nyo Me Htwe; Nazira Q. Kama...,NaN,NaN,False,Country / Area
31,104,Myanmar,NaN,23161.02,"Rice, milled",2015,17.65,17.65%,132.27kgs,Storage,Storage,"90 days storage, no trapping",Rodents,NaN,Controlled Experiment,Steven R. Belmain; Nyo Me Htwe; Nazira Q. Kama...,NaN,NaN,False,Country / Area
32,104,Myanmar,NaN,23161.02,"Rice, milled",2015,3.57,3.57%,52.24kgs,Storage,Storage,"60 days storage, with trapping",Rodents,NaN,Controlled Experiment,Steven R. Belmain; Nyo Me Htwe; Nazira Q. Kama...,NaN,NaN,False,Country / Area
33,104,Myanmar,NaN,23161.02,"Rice, milled",2015,5.35,5.35%,78.36kgs,Storage,Storage,"90 days storage, with trapping",Rodents,NaN,Controlled Experiment,Steven R. Belmain; Nyo Me Htwe; Nazira Q. Kama...,NaN,NaN,False,Country / Area


In [44]:
# [ADDED] sanity check: only Israel + aggregates (World / regions) should disappear
removed = data_frame.loc[~data_frame["m49_code"].isin(df["m49_code"]), ["m49_code", "country"]].drop_duplicates()
display(removed.sort_values("m49_code"))

,m49_code,country
0,1,World
2750,2,Africa
551,11,Western Africa
1939,15,Northern Africa
3562,21,Northern America
8757,34,Southern Asia
9063,35,South-Eastern Asia
18013,53,Australia and New Zealand
1292,142,Asia
1293,143,Central Asia


## **2. region column**

In [45]:
# Drop region-related columns safely

columns_to_drop = [
    "region",
    "region_type",
    "region_type_review",
    "automatic_region_type",
    "reviewed_region_type"
]

df.drop(
    columns=columns_to_drop,
    inplace=True,
    errors="ignore"
)

# Verify removal
remaining_columns = [
    col for col in columns_to_drop if col in df.columns
]

print("Columns still present:", remaining_columns)
print("All specified columns removed:", len(remaining_columns) == 0)

Columns still present: []
All specified columns removed: True


## **3. cpc_code column**

### **1. CPC code consistent with commmity**

In [46]:

# ==================================================
# 1. Remove unwanted columns
# ==================================================
df.drop(
    columns=["commodity_original", "commodity_normalized"],
    errors="ignore",
    inplace=True
)

# ==================================================
# 2. Define canonical commodity names
# ==================================================
canonical_commodity_names = {
    "01199.90": "Other cereals n.e.c.",
    "01231": "Chillies and peppers, green (Capsicum spp. and Pimenta spp.)",
    "01290.90": "Other vegetables, fresh n.e.c.",
    "01313": "Plantains and cooking bananas",
    "01316": "Mangoes, guavas and mangosteens",
    "01319": "Other tropical and subtropical fruits, n.e.c.",
    "01329": "Other citrus fruit, n.e.c.",
    "01353.01": "Raspberries",
    "01355.02": "Cranberries",
    "01359.90": "Other fruits, n.e.c.",
    "01443": "Rapeseed or colza seed",
    "01599.10": "Edible roots and tubers with high starch or inulin content, n.e.c., fresh",
    "01709.90": "Other pulses n.e.c.",
    "21111.01": "Meat of cattle with the bone, fresh or chilled",
    "21113.01": "Meat of pig with the bone, fresh or chilled",
    "22251.01": "Cheese from whole cow milk",
    "23161.02": "Rice, milled"
}

# ==================================================
# 3. Standardize CPC codes and commodity names
# ==================================================
df["cpc_code"] = df["cpc_code"].astype("string").str.strip()

target_mask = df["cpc_code"].isin(canonical_commodity_names)

df.loc[target_mask, "commodity"] = (
    df.loc[target_mask, "cpc_code"]
    .map(canonical_commodity_names)
)

# ==================================================
# 4. Validate all 17 CPC codes
# ==================================================
validation = (
    df[df["cpc_code"].isin(canonical_commodity_names)]
    .groupby("cpc_code")
    .agg(
        commodity_count=("commodity", "nunique"),
        commodities=(
            "commodity",
            lambda s: sorted(s.dropna().unique().tolist())
        ),
        records=("commodity", "size")
    )
    .reindex(list(canonical_commodity_names))
    .rename_axis("cpc_code")
    .reset_index()
)

validation["expected_name"] = validation["cpc_code"].map(
    canonical_commodity_names
)

validation["is_consistent"] = (
    validation["commodity_count"].eq(1)
    & validation["commodities"].apply(
        lambda names: len(names) == 1
    )
)

validation["name_matches_expected"] = validation.apply(
    lambda row: (
        row["commodities"] == [row["expected_name"]]
    ),
    axis=1
)

# ==================================================
# 5. Check missing CPC codes
# ==================================================
missing_codes = [
    code for code in canonical_commodity_names
    if not df["cpc_code"].eq(code).any()
]

# ==================================================
# 6. Check remaining inconsistencies
# ==================================================
remaining_inconsistencies = (
    df.groupby("cpc_code", dropna=False)
    .agg(
        commodity_count=("commodity", "nunique"),
        commodities=(
            "commodity",
            lambda s: sorted(s.dropna().unique().tolist())
        ),
        records=("commodity", "size")
    )
    .reset_index()
    .query("commodity_count > 1")
)

# ==================================================
# 7. Check unwanted columns
# ==================================================
unwanted_columns = [
    col for col in ["commodity_original", "commodity_normalized"]
    if col in df.columns
]

# ==================================================
# 8. Final verification
# ==================================================
all_codes_present = len(missing_codes) == 0

all_names_consistent = validation["is_consistent"].all()

all_names_correct = validation["name_matches_expected"].all()

no_unwanted_columns = len(unwanted_columns) == 0

success = (
    all_codes_present
    and all_names_consistent
    and all_names_correct
    and no_unwanted_columns
)

# ==================================================
# 9. Display validation results
# ==================================================
print("=" * 60)
print("FINAL CPC STANDARDIZATION VALIDATION")
print("=" * 60)

print(f"Expected CPC codes: {len(canonical_commodity_names)}")
print(f"Validated CPC codes: {len(validation)}")
print(f"Missing CPC codes: {missing_codes}")
print(f"Remaining inconsistent CPC codes: {len(remaining_inconsistencies)}")
print(f"Unwanted columns: {unwanted_columns}")

display(validation)

if not remaining_inconsistencies.empty:
    print("Remaining inconsistencies across the entire dataset:")
    display(remaining_inconsistencies)


FINAL CPC STANDARDIZATION VALIDATION
Expected CPC codes: 17
Validated CPC codes: 17
Missing CPC codes: []
Remaining inconsistent CPC codes: 0
Unwanted columns: []


,cpc_code,commodity_count,commodities,records,expected_name,is_consistent,name_matches_expected
0,01199.90,1,[Other cereals n.e.c.],27,Other cereals n.e.c.,True,True
1,01231,1,"[Chillies and peppers, green (Capsicum spp. an...",57,"Chillies and peppers, green (Capsicum spp. and...",True,True
2,01290.90,1,"[Other vegetables, fresh n.e.c.]",52,"Other vegetables, fresh n.e.c.",True,True
3,01313,1,[Plantains and cooking bananas],42,Plantains and cooking bananas,True,True
4,01316,1,"[Mangoes, guavas and mangosteens]",301,"Mangoes, guavas and mangosteens",True,True
5,01319,1,"[Other tropical and subtropical fruits, n.e.c.]",56,"Other tropical and subtropical fruits, n.e.c.",True,True
6,01329,1,"[Other citrus fruit, n.e.c.]",37,"Other citrus fruit, n.e.c.",True,True
7,01353.01,1,[Raspberries],36,Raspberries,True,True
8,01355.02,1,[Cranberries],35,Cranberries,True,True
9,01359.90,1,"[Other fruits, n.e.c.]",77,"Other fruits, n.e.c.",True,True


## **4. commitiy column**

### **1. Handle diff names for same meaning**

In [47]:
# Standardize commodity names directly in df
commodity_corrections = {
    "22251.01": "Cheese from whole cow milk",
    "23161.02": "Rice, milled"
}

df["cpc_code"] = (
    df["cpc_code"].astype("string").str.strip()
)

mask = df["cpc_code"].isin(commodity_corrections)

df.loc[mask, "commodity"] = (
    df.loc[mask, "cpc_code"]
    .map(commodity_corrections)
)

In [48]:
# 1. Normalize names for comparison only
commodity_normalized = (
    df["commodity"]
    .astype("string")
    .str.strip()
    .str.casefold()
    .str.replace(r"\s+", " ", regex=True)
)

# 2. Find different original names with the same normalized name
commodity_variants = (
    df.assign(_commodity_normalized=commodity_normalized)
    .dropna(subset=["_commodity_normalized"])
    .groupby("_commodity_normalized")
    .agg(
        original_names=(
            "commodity",
            lambda s: sorted(s.astype(str).unique())
        ),
        records=("commodity", "size"),
        cpc_codes=(
            "cpc_code",
            lambda s: sorted(s.dropna().astype(str).unique())
        )
    )
    .reset_index()
)

commodity_variants = commodity_variants.loc[
    commodity_variants["original_names"].apply(len) > 1
].copy()

# 3. Display results
print("Names with formatting/case variants:", len(commodity_variants))

display(commodity_variants)

# 4. Final check
if commodity_variants.empty:
    print("SUCCESS: No formatting or capitalization inconsistencies found!")
else:
    print("WARNING: Formatting or capitalization inconsistencies still exist.")

Names with formatting/case variants: 0


,_commodity_normalized,original_names,records,cpc_codes


SUCCESS: No formatting or capitalization inconsistencies found!


In [49]:
# Remove HTML tags (e.g. <i>Capsicum</i>) from commodity names
df["commodity"] = (
    df["commodity"]
    .astype("string")
    .str.replace(r"</?[a-zA-Z]+>", "", regex=True)   # remove <i> </i> and any similar tag
    .str.replace(r"\s+", " ", regex=True)            # fix double spaces left behind
    .str.strip()
)

# Check
print("Remaining HTML tags:", df["commodity"].str.contains(r"<|>", regex=True).sum())
print("CPC codes with more than one name:", (df.groupby("cpc_code")["commodity"].nunique() > 1).sum())

Remaining HTML tags: 0
CPC codes with more than one name: 0


### **2. Handle spelling**

In [50]:


# ==================================================
# 1. Initialize English spell checker
# ==================================================

spell = SpellChecker(language="en")


# ==================================================
# 2. Add valid agricultural and scientific terms
# ==================================================

technical_words = {
    "fonio",
    "pimenta",
    "spp",
    "triticale",
    "meslin",
    "badian",
    "quinoa",
    "vaccinium",
    "unginned",
    "unrendered",
    "capsicum",
    "piper"
}

spell.word_frequency.load_words(technical_words)


# ==================================================
# 3. Identify spelling corrections
# ==================================================

def correct_spelling(value):

    if pd.isna(value):
        return value

    original_value = str(value)

    words = re.findall(r"[A-Za-z]+", original_value)

    corrections = {}

    for word in words:

        word_lower = word.lower()

        if word_lower in technical_words:
            continue

        if word_lower in spell:
            continue

        candidates = spell.candidates(word_lower)

        if candidates:
            suggestions = sorted(candidates)

            if len(suggestions) == 1:
                corrections[word] = suggestions[0]

    corrected_value = original_value

    for wrong_word, correct_word in corrections.items():

        corrected_value = re.sub(
            rf"\b{re.escape(wrong_word)}\b",
            correct_word,
            corrected_value,
            flags=re.IGNORECASE
        )

    return corrected_value


# ==================================================
# 4. Apply corrections directly to df
# ==================================================

df["commodity"] = df["commodity"].apply(correct_spelling)


# ==================================================
# 5. Display changed commodity names
# ==================================================

print("Spelling correction completed.")

display(df["commodity"].value_counts().head(30))

Spelling correction completed.


commodity
Maize (corn)                       5948
Rice                               4753
Millet                             3572
Sorghum                            3441
Wheat                              2354
Fonio                               968
Barley                              852
Potatoes                            648
Tomatoes                            479
Mangoes, guavas and mangosteens     301
Cabbages                            237
Grapes                              221
Oats                                215
Hen eggs in shell, fresh            192
Bananas                             186
Beans, dry                          184
Cauliflowers and broccoli           167
Apples                              163
Rye                                 146
Carrots and turnips                 144
Onions and shallots, green          141
Raw milk of cattle                  132
Papayas                             132
Pineapples                          128
Soya beans                    

In [51]:
df.head()

,m49_code,country,cpc_code,commodity,year,loss_percentage,loss_percentage_original,loss_quantity,activity,food_supply_stage,treatment,cause_of_loss,sample_size,method_data_collection,reference,url,notes,loss_is_range,record_type
29,104,Myanmar,23161.02,"Rice, milled",2015,5.88,5.88%,44.09kgs,Storage,Storage,"30 days storage, no trapping",Rodents,NaN,Controlled Experiment,Steven R. Belmain; Nyo Me Htwe; Nazira Q. Kama...,NaN,NaN,False,Country / Area
30,104,Myanmar,23161.02,"Rice, milled",2015,11.77,11.77%,88.18kgs,Storage,Storage,"60 days storage, no trapping",Rodents,NaN,Controlled Experiment,Steven R. Belmain; Nyo Me Htwe; Nazira Q. Kama...,NaN,NaN,False,Country / Area
31,104,Myanmar,23161.02,"Rice, milled",2015,17.65,17.65%,132.27kgs,Storage,Storage,"90 days storage, no trapping",Rodents,NaN,Controlled Experiment,Steven R. Belmain; Nyo Me Htwe; Nazira Q. Kama...,NaN,NaN,False,Country / Area
32,104,Myanmar,23161.02,"Rice, milled",2015,3.57,3.57%,52.24kgs,Storage,Storage,"60 days storage, with trapping",Rodents,NaN,Controlled Experiment,Steven R. Belmain; Nyo Me Htwe; Nazira Q. Kama...,NaN,NaN,False,Country / Area
33,104,Myanmar,23161.02,"Rice, milled",2015,5.35,5.35%,78.36kgs,Storage,Storage,"90 days storage, with trapping",Rodents,NaN,Controlled Experiment,Steven R. Belmain; Nyo Me Htwe; Nazira Q. Kama...,NaN,NaN,False,Country / Area


## **5. Activity**

### **1. Handle missing values**

In [52]:
# [ADDED] activity handling (replaces the explode + ML cells above)
# 5,939 of the 7,370 missing activities are "Whole supply chain" rows: the loss covers the whole chain,
# so activity is not unknown, it is the whole chain. food_supply_stage tells us that.
df["record_id"] = np.arange(1, len(df) + 1)

def normalize_activity(value):
    if pd.isna(value):
        return pd.NA
    parts = sorted({p.strip() for p in str(value).split(",") if p.strip()})
    return ", ".join(parts) if parts else pd.NA

activity = df["activity"].astype("string").map(normalize_activity).astype("string")

# Only stages whose activity matches in >= 90% of rows that have both columns
stage_to_activity = {
    "Whole supply chain": "Whole supply chain",
    "Storage": "Storage",
    "Transport": "Transportation",
    "Retail": "Retailing",
    "Harvest": "Harvesting",
    "Packing": "Packaging",
    "Grading": "Grading",
    "Distribution": "Distribution",
}

df["activity_source"] = "Original"
missing = activity.isna()
filled = df.loc[missing, "food_supply_stage"].map(stage_to_activity)
activity.loc[missing] = filled.astype("string")
df.loc[missing & filled.notna(), "activity_source"] = "Filled from food_supply_stage"
df.loc[activity.isna(), "activity_source"] = "Unknown"

df["activity"] = activity.fillna("Unknown")
df["activity_count"] = np.where(df["activity"].eq("Unknown"), 0, df["activity"].str.count(",") + 1)

display(df["activity_source"].value_counts())

activity_source
Original                         23518
Filled from food_supply_stage     6355
Unknown                            717
Name: count, dtype: int64

In [53]:
# [ADDED] one row per (record, activity) for activity-level analysis only.
# Do NOT average loss_percentage on this table: one measurement would be counted several times.
df_activity_long = df[["record_id", "activity"]].copy()
df_activity_long["activity"] = df_activity_long["activity"].str.split(", ")
df_activity_long = df_activity_long.explode("activity")
df_activity_long.to_csv("../report/activity/World-food-waste_activity_long.csv", index=False)
print(df_activity_long.shape)

(39651, 2)


In [54]:
df.head()

,m49_code,country,cpc_code,commodity,year,loss_percentage,loss_percentage_original,loss_quantity,activity,food_supply_stage,...,sample_size,method_data_collection,reference,url,notes,loss_is_range,record_type,record_id,activity_source,activity_count
29,104,Myanmar,23161.02,"Rice, milled",2015,5.88,5.88%,44.09kgs,Storage,Storage,...,NaN,Controlled Experiment,Steven R. Belmain; Nyo Me Htwe; Nazira Q. Kama...,NaN,NaN,False,Country / Area,1,Original,1
30,104,Myanmar,23161.02,"Rice, milled",2015,11.77,11.77%,88.18kgs,Storage,Storage,...,NaN,Controlled Experiment,Steven R. Belmain; Nyo Me Htwe; Nazira Q. Kama...,NaN,NaN,False,Country / Area,2,Original,1
31,104,Myanmar,23161.02,"Rice, milled",2015,17.65,17.65%,132.27kgs,Storage,Storage,...,NaN,Controlled Experiment,Steven R. Belmain; Nyo Me Htwe; Nazira Q. Kama...,NaN,NaN,False,Country / Area,3,Original,1
32,104,Myanmar,23161.02,"Rice, milled",2015,3.57,3.57%,52.24kgs,Storage,Storage,...,NaN,Controlled Experiment,Steven R. Belmain; Nyo Me Htwe; Nazira Q. Kama...,NaN,NaN,False,Country / Area,4,Original,1
33,104,Myanmar,23161.02,"Rice, milled",2015,5.35,5.35%,78.36kgs,Storage,Storage,...,NaN,Controlled Experiment,Steven R. Belmain; Nyo Me Htwe; Nazira Q. Kama...,NaN,NaN,False,Country / Area,5,Original,1


In [55]:
# List unique activity labels and their frequencies
activity_frequency = (
    df["activity"]
    .dropna()
    .astype("string")
    .str.strip()
    .value_counts()
    .rename_axis("activity")
    .reset_index(name="records")
)

print("Unique non-empty activity labels:", len(activity_frequency))

activity_frequency.to_csv(
    "../report/activity/potential_synonyms_after_split.csv"
)

display(activity_frequency)

Unique non-empty activity labels: 179


,activity,records
0,Storage,6208
1,Whole supply chain,5686
2,Transportation,5275
3,"Drying, Harvesting",3412
4,"Shelling, Threshing",3258
...,...,...
174,"Drying, Harvesting, Milling, Sorting, Storage,...",1
175,"Sorting, Washing",1
176,"Milling, Storage",1
177,"Harvesting, Packaging, Sorting, Storage",1


## **6. "loss percentage original" , "loss_quantity",  "treatment",	"cause_of_loss"	,"sample_size" , "notes"  ,  "food_supply_stage" column**

In [56]:
df = df.drop(columns=["loss_percentage_original" , "loss_quantity" , "treatment",	"cause_of_loss"	,"sample_size" , "notes" , "food_supply_stage"])

## **Finish step Check on the data-set**

In [57]:
df.shape

(30590, 15)

In [58]:
print("=" * 60)
print("DATASET OVERVIEW")
print("=" * 60)

print("Shape:", df.shape)
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])
print("Memory usage (MB):", round(df.memory_usage(deep=True).sum() / (1024 ** 2), 2))

display(df.sample(min(5, len(df)), random_state=42))

DATASET OVERVIEW
Shape: (30590, 15)
Number of rows: 30590
Number of columns: 15
Memory usage (MB): 20.0


,m49_code,country,cpc_code,commodity,year,loss_percentage,activity,method_data_collection,reference,url,loss_is_range,record_type,record_id,activity_source,activity_count
12636,404,Kenya,0112,Maize (corn),2008,1.65000,Transportation,Modelled Estimates,"APHLIS, (2021), APHLIS",https://www.aphlis.net/en/data/tables/value-ch...,False,Country / Area,12064,Original,1
29602,854,Burkina Faso,0112,Maize (corn),2015,7.21802,Storage,Modelled Estimates,"APHLIS, (2021), APHLIS",https://www.aphlis.net/en/data/tables/value-ch...,False,Country / Area,28922,Original,1
12968,404,Kenya,0118,Millet,2000,2.50000,Transportation,Modelled Estimates,"APHLIS, (2021), APHLIS",https://www.aphlis.net/en/data/tables/value-ch...,False,Country / Area,12396,Original,1
12435,404,Kenya,0113,Rice,2012,4.40333,"Drying, Harvesting",Modelled Estimates,"APHLIS, (2021), APHLIS",https://www.aphlis.net/en/data/tables/value-ch...,False,Country / Area,11863,Original,2
22971,706,Somalia,0112,Maize (corn),2001,2.41667,Transportation,Modelled Estimates,"APHLIS, (2021), APHLIS",https://www.aphlis.net/en/data/tables/value-ch...,False,Country / Area,22325,Original,1


In [59]:
print("COLUMN NAMES")
print(df.columns.tolist())

print("\nDATA TYPES")
print(df.dtypes)

print("\nDATASET INFO")
df.info()

print("\nNUMERIC SUMMARY")
display(df.describe().T)

print("\nALL COLUMNS SUMMARY")
display(df.describe(include="all").T)

COLUMN NAMES
['m49_code', 'country', 'cpc_code', 'commodity', 'year', 'loss_percentage', 'activity', 'method_data_collection', 'reference', 'url', 'loss_is_range', 'record_type', 'record_id', 'activity_source', 'activity_count']

DATA TYPES
m49_code                    int64
country                       str
cpc_code                   string
commodity                     str
year                        int64
loss_percentage           float64
activity                   string
method_data_collection        str
reference                     str
url                           str
loss_is_range             boolean
record_type                   str
record_id                   int64
activity_source               str
activity_count              int64
dtype: object

DATASET INFO
<class 'pandas.DataFrame'>
Index: 30590 entries, 29 to 31270
Data columns (total 15 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   m49_code        

,count,mean,std,min,25%,50%,75%,max
m49_code,30590.0,472.230631,261.634452,4.0,232.00000,454.00,710.00,894.0
year,30590.0,2010.599150,6.215976,2000.0,2005.00000,2011.00,2016.00,2026.0
loss_percentage,30590.0,4.670690,6.382618,0.0,1.47563,2.65,4.55,65.0
record_id,30590.0,15295.500000,8830.716704,1.0,7648.25000,15295.50,22942.75,30590.0
activity_count,30590.0,1.272769,0.790112,0.0,1.00000,1.00,1.00,11.0



ALL COLUMNS SUMMARY


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
m49_code,30590.0,NaN,NaN,NaN,472.230631,261.634452,4.0,232.0,454.0,710.0,894.0
country,30590,136,USA,1575,NaN,NaN,NaN,NaN,NaN,NaN,NaN
cpc_code,30590,196,0112,5948,NaN,NaN,NaN,NaN,NaN,NaN,NaN
commodity,30590,196,Maize (corn),5948,NaN,NaN,NaN,NaN,NaN,NaN,NaN
year,30590.0,NaN,NaN,NaN,2010.59915,6.215976,2000.0,2005.0,2011.0,2016.0,2026.0
loss_percentage,30590.0,NaN,NaN,NaN,4.67069,6.382618,0.0,1.47563,2.65,4.55,65.0
activity,30590,179,Storage,6208,NaN,NaN,NaN,NaN,NaN,NaN,NaN
method_data_collection,30554,11,Modelled Estimates,19023,NaN,NaN,NaN,NaN,NaN,NaN,NaN
reference,30517,320,"APHLIS, (2021), APHLIS",18504,NaN,NaN,NaN,NaN,NaN,NaN,NaN
url,21538,925,https://www.ers.usda.gov/data-products/food-av...,1057,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [60]:
missing_report = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percentage": df.isna().mean().mul(100),
    "non_missing_count": df.notna().sum(),
    "unique_non_null": df.nunique(dropna=True)
})

missing_report = missing_report.sort_values(
    by="missing_percentage",
    ascending=False
)

display(missing_report)

,missing_count,missing_percentage,non_missing_count,unique_non_null
url,9052,29.591370,21538,925
reference,73,0.238640,30517,320
method_data_collection,36,0.117686,30554,11
commodity,0,0.000000,30590,196
m49_code,0,0.000000,30590,136
country,0,0.000000,30590,136
cpc_code,0,0.000000,30590,196
activity,0,0.000000,30590,179
loss_percentage,0,0.000000,30590,4603
year,0,0.000000,30590,27


In [61]:
unique_report = pd.DataFrame({
    "unique_count": df.nunique(dropna=True),
    "unique_with_nan": df.nunique(dropna=False),
    "dtype": df.dtypes.astype(str),
    "is_constant": df.nunique(dropna=True).le(1)
})

display(unique_report.sort_values("unique_count"))

,unique_count,unique_with_nan,dtype,is_constant
record_type,1,1,str,True
loss_is_range,2,2,boolean,False
activity_source,3,3,str,False
method_data_collection,11,12,str,False
activity_count,12,12,int64,False
year,27,27,int64,False
m49_code,136,136,int64,False
country,136,136,str,False
activity,179,179,string,False
cpc_code,196,196,string,False


## **final step save the final data-set**

In [62]:
df = df.drop(columns=["record_type"])

In [63]:
df.to_csv("../data-set/World-food-waste_cleaned_final.csv", index=False)  # [CHANGED] index=False, new name